In [ ]:
# Task 1: Load Crop Recommendation Dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load only DF_2 (Crop Recommendation Dataset)
df = pd.read_csv('DF_2_Crop_recommendation.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumn Names: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nDataset Info:")
print(df.info())
print(f"\nUnique Crops: {df['label'].nunique()}")
print(f"Crop Types: {sorted(df['label'].unique())}")



In [ ]:
# Task 2: Data Preprocessing and Cleaning
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Check for null values
print("Null values per column:")
print(df.isnull().sum())
print(f"\nTotal null values: {df.isnull().sum().sum()}")

# Remove null values if any
df_clean = df.dropna().copy()
print(f"\nDataset shape after removing nulls: {df_clean.shape}")

# Check for duplicates
print(f"\nDuplicate rows: {df_clean.duplicated().sum()}")
df_clean = df_clean.drop_duplicates().copy()
print(f"Dataset shape after removing duplicates: {df_clean.shape}")

# Separate features and target
# Features: N, P, K, temperature, humidity, ph, rainfall
X = df_clean[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']].copy()
# Target: label (crop name)
y = df_clean['label'].copy()

print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())



In [ ]:
# Task 3: Label Encoding for Target Variable
# Encode crop names to numeric labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Original crop names: {y.unique()[:5]}...")
print(f"Encoded labels: {y_encoded[:5]}...")
print(f"\nNumber of unique crops: {len(label_encoder.classes_)}")
print(f"Crop classes: {label_encoder.classes_}")

# Create mapping for reference
crop_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))
print(f"\nCrop to Label Mapping (first 5):")
for i, (crop, label) in enumerate(list(crop_mapping.items())[:5]):
    print(f"  {crop}: {label}")



In [ ]:
# Task 4: Feature Scaling and Train-Test Split
# Scale features for better model performance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Feature scaling completed.")
print(f"Scaled features shape: {X_scaled.shape}")
print(f"\nScaled feature statistics:")
print(X_scaled.describe())

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")
print(f"\nTraining set class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for label, count in zip(unique[:5], counts[:5]):
    crop_name = label_encoder.inverse_transform([label])[0]
    print(f"  {crop_name}: {count} samples")



In [ ]:
# Task 5: Train Classification Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import time

# Initialize models
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
}

# Train and evaluate models
trained_models = {}
results = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training {name}...")
    print(f"{'='*60}")
    
    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Predictions
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    # Calculate metrics
    train_accuracy = accuracy_score(y_train, train_pred)
    test_accuracy = accuracy_score(y_test, test_pred)
    
    trained_models[name] = model
    results[name] = {
        'train_accuracy': train_accuracy,
        'test_accuracy': test_accuracy,
        'training_time': training_time
    }
    
    print(f"Training Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
    print(f"Testing Accuracy:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
    print(f"Training Time: {training_time:.2f} seconds")
    
    # Classification report
    print(f"\nClassification Report:")
    print(classification_report(y_test, test_pred, 
                                target_names=label_encoder.classes_,
                                zero_division=0))

# Compare models
print(f"\n{'='*60}")
print("MODEL COMPARISON SUMMARY")
print(f"{'='*60}")
for name, metrics in results.items():
    print(f"{name:20s} | Test Accuracy: {metrics['test_accuracy']:.4f} | Time: {metrics['training_time']:.2f}s")

# Select best model (highest test accuracy)
best_model_name = max(results.keys(), key=lambda x: results[x]['test_accuracy'])
best_model = trained_models[best_model_name]
print(f"\nBest Model: {best_model_name} (Test Accuracy: {results[best_model_name]['test_accuracy']:.4f})")



In [ ]:
# Task 6: Create Function to Get Top 3 Crop Recommendations
def get_top_3_crops(model, label_encoder, scaler, N, P, K, temperature, humidity, ph, rainfall):
    """
    Predict top 3 suitable crops based on soil and environmental parameters.
    
    Parameters:
    -----------
    model : Trained classifier model
    label_encoder : LabelEncoder fitted on crop names
    scaler : StandardScaler fitted on training data
    N, P, K : Soil nutrients (ppm)
    temperature : Temperature (°C)
    humidity : Humidity (%)
    ph : pH level
    rainfall : Rainfall (mm)
    
    Returns:
    --------
    list : List of tuples (crop_name, probability) sorted by probability (descending)
    """
    # Create input array
    input_data = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
    
    # Scale the input
    input_scaled = scaler.transform(input_data)
    
    # Get probability predictions
    probabilities = model.predict_proba(input_scaled)[0]
    
    # Get class indices sorted by probability (descending)
    top_indices = np.argsort(probabilities)[::-1][:3]
    
    # Get top 3 crops with their probabilities
    top_3_crops = []
    for idx in top_indices:
        crop_name = label_encoder.inverse_transform([idx])[0]
        probability = probabilities[idx]
        top_3_crops.append((crop_name, probability))
    
    return top_3_crops

# Test the function with sample inputs
print("Testing Top 3 Crop Recommendation Function:")
print("="*60)

# Test Case 1: High rainfall, suitable for rice
print("\nTest Case 1: High Rainfall Conditions")
print("Input: N=90, P=42, K=43, Temp=20.8°C, Humidity=82%, pH=6.5, Rainfall=202.9mm")
top_3 = get_top_3_crops(best_model, label_encoder, scaler, 
                        N=90, P=42, K=43, temperature=20.8, 
                        humidity=82.0, ph=6.5, rainfall=202.9)
for i, (crop, prob) in enumerate(top_3, 1):
    print(f"  {i}. {crop.capitalize()}: {prob:.4f} ({prob*100:.2f}%)")

# Test Case 2: Dry conditions
print("\nTest Case 2: Dry Conditions")
print("Input: N=20, P=40, K=20, Temp=25.0°C, Humidity=40%, pH=7.0, Rainfall=30.0mm")
top_3 = get_top_3_crops(best_model, label_encoder, scaler,
                        N=20, P=40, K=20, temperature=25.0,
                        humidity=40.0, ph=7.0, rainfall=30.0)
for i, (crop, prob) in enumerate(top_3, 1):
    print(f"  {i}. {crop.capitalize()}: {prob:.4f} ({prob*100:.2f}%)")

# Test Case 3: Moderate conditions
print("\nTest Case 3: Moderate Conditions")
print("Input: N=50, P=50, K=50, Temp=22.0°C, Humidity=60%, pH=7.0, Rainfall=100.0mm")
top_3 = get_top_3_crops(best_model, label_encoder, scaler,
                        N=50, P=50, K=50, temperature=22.0,
                        humidity=60.0, ph=7.0, rainfall=100.0)
for i, (crop, prob) in enumerate(top_3, 1):
    print(f"  {i}. {crop.capitalize()}: {prob:.4f} ({prob*100:.2f}%)")



In [ ]:
# Task 7: Save Model and Encoder Artifacts
import joblib

# Save the best model
joblib.dump(best_model, 'crop_classifier_model.pkl')
print(f"✓ Saved: crop_classifier_model.pkl ({best_model_name})")

# Save the label encoder (for decoding predictions back to crop names)
joblib.dump(label_encoder, 'crop_name_encoder.pkl')
print(f"✓ Saved: crop_name_encoder.pkl")

# Also save the scaler for consistency (optional, but useful)
joblib.dump(scaler, 'crop_scaler.pkl')
print(f"✓ Saved: crop_scaler.pkl (for feature scaling)")

print(f"\n{'='*60}")
print("All artifacts saved successfully!")
print(f"{'='*60}")
print(f"\nModel Summary:")
print(f"  Model Type: {best_model_name}")
print(f"  Test Accuracy: {results[best_model_name]['test_accuracy']:.4f} ({results[best_model_name]['test_accuracy']*100:.2f}%)")
print(f"  Number of Classes: {len(label_encoder.classes_)}")
print(f"  Features: N, P, K, temperature, humidity, ph, rainfall")



In [ ]:
# Task 8: Visualization - Confusion Matrix and Feature Importance
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Confusion Matrix
y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            cbar_kws={'label': 'Count'})
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Crop', fontsize=12)
plt.ylabel('Actual Crop', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Feature Importance (if available)
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': best_model.feature_importances_
    }).sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Feature', data=feature_importance, palette='viridis')
    plt.title(f'Feature Importance - {best_model_name}', fontsize=14, fontweight='bold')
    plt.xlabel('Importance Score', fontsize=12)
    plt.ylabel('')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\nFeature Importance Ranking:")
    print(feature_importance.to_string(index=False))

